# Classification du prix des téléphones mobiles - Analyse statistique complète

**Objectif :** explorer le jeu de données *Mobile Price Classification*, mener une analyse
statistique complète avec **NumPy** et **SciPy**, le visualiser avec **Matplotlib**, et
identifier quelles caractéristiques déterminent la **gamme de prix**
(0 = bas, 1 = moyen, 2 = élevé, 3 = très élevé).

*Cette copie est commentée fonction par fonction pour la compréhension.*

In [ ]:
# === IMPORTS : on charge les bibliothèques nécessaires ===
import numpy as np                 # np : calcul numérique (tableaux, statistiques)
import pandas as pd                # pd : tableaux de données (le "df")
import matplotlib.pyplot as plt    # plt : graphiques
from scipy import stats            # stats : tests statistiques (ANOVA, test t, corrélation)

## 1. Chargement et exploration des données

In [ ]:
# pd.read_csv : lit le fichier CSV et le transforme en DataFrame (un tableau)
df = pd.read_csv("train.csv")

# df.head() : affiche les 5 premières lignes pour vérifier la structure
df.head()

In [ ]:
# df.shape : renvoie un couple (nombre de lignes, nombre de colonnes)
print("Shape:", df.shape)

# df.dtypes : le type de données de chaque colonne (int64 = entier, float64 = décimal)
print("\nData types:")
print(df.dtypes)

In [ ]:
# df.describe() : statistiques de base de chaque colonne numérique
#   count = nombre de valeurs, mean = moyenne, std = écart-type,
#   min/max = bornes, 25%/50%/75% = quartiles (50% = médiane)
df.describe()

In [ ]:
# df["price_range"] : on sélectionne la colonne cible
# .value_counts() : compte combien de fois apparaît chaque valeur
# .sort_index() : trie par classe (0,1,2,3) pour une lecture claire
# Rappel : 0 = bas, 1 = moyen, 2 = élevé, 3 = très élevé
df["price_range"].value_counts().sort_index()

## 2. Nettoyage et prétraitement des données

In [ ]:
# df.isnull() : renvoie True/False pour chaque case vide
# .sum() : additionne les True (= 1) par colonne -> nombre de valeurs manquantes
df.isnull().sum()

In [ ]:
# df.duplicated() : True pour chaque ligne identique à une précédente
# .sum() : nombre total de lignes en double
print("Duplicated rows:", df.duplicated().sum())

# df.select_dtypes(include="object") : sélectionne les colonnes de type texte
# .columns : la liste de leurs noms (vide ici -> aucune colonne à encoder)
print("Text (object) columns:", list(df.select_dtypes(include="object").columns))

In [ ]:
# Toutes les colonnes sont déjà numériques et il n'y a aucune valeur manquante :
# aucun remplissage ni encodage n'est nécessaire.
# Les variables binaires (blue, dual_sim, four_g, three_g, touch_screen, wifi)
# sont déjà encodées en 0/1.

# price_labels : un dictionnaire qui associe chaque code à un libellé lisible
price_labels = {0: "Low", 1: "Medium", 2: "High", 3: "Very High"}
# .map(price_labels) : remplace chaque code par son libellé -> nouvelle colonne texte
df["price_label"] = df["price_range"].map(price_labels)
df[["price_range", "price_label"]].head()

## 3. Analyse statistique avec NumPy et SciPy

In [ ]:
# continuous : la liste des variables continues à analyser en détail
continuous = ["battery_power", "clock_speed", "fc", "int_memory", "m_dep",
              "mobile_wt", "n_cores", "pc", "px_height", "px_width", "ram",
              "sc_h", "sc_w", "talk_time"]

# Astuce : [ f(c) for c in continuous ] = "compréhension de liste"
#   -> calcule f pour CHAQUE variable de la liste, d'un coup.

# pd.DataFrame(index=continuous) : un tableau vide, une ligne par variable
summary = pd.DataFrame(index=continuous)
# np.mean : la moyenne (somme / nombre de valeurs)
summary["mean"]   = [np.mean(df[c]) for c in continuous]
# np.median : la valeur du milieu (50% au-dessus, 50% en dessous)
summary["median"] = [np.median(df[c]) for c in continuous]
# .mode() : la valeur la plus fréquente ; .iloc[0] : on garde la première
summary["mode"]   = [df[c].mode().iloc[0] for c in continuous]
summary.round(2)   # .round(2) : arrondi à 2 décimales pour l'affichage

In [ ]:
# .max() - .min() : l'étendue (range) = écart entre la plus grande et la plus petite valeur
summary["range"]    = [df[c].max() - df[c].min() for c in continuous]
# np.var(..., ddof=1) : la variance d'échantillon (ddof=1 = division par n-1)
summary["variance"] = [np.var(df[c], ddof=1) for c in continuous]
# np.std(..., ddof=1) : l'écart-type d'échantillon (= racine carrée de la variance)
summary["std"]      = [np.std(df[c], ddof=1) for c in continuous]
summary.round(2)

In [ ]:
# stats.skew : l'asymétrie de la distribution
#   ~ 0 = symétrique, > 0 = queue à droite, < 0 = queue à gauche
summary["skew"]     = [stats.skew(df[c]) for c in continuous]
# stats.kurtosis : l'aplatissement (excès -> loi normale = 0)
#   > 0 = pic + queues épaisses, < 0 = distribution plate
summary["kurtosis"] = [stats.kurtosis(df[c]) for c in continuous]
summary.round(2)

**Lecture du tableau :** la plupart des variables sont quasi *uniformes* (skew ~ 0 et
kurtosis négatif ~ -1,2, donc des distributions plates). Quelques-unes sont asymétriques à
droite : caméra frontale (`fc`), hauteur en pixels (`px_height`) et largeur d'écran (`sc_w`).

In [ ]:
# TEST D'HYPOTHÈSE 1 - ANOVA (compare des moyennes entre PLUSIEURS groupes)
# H0 : la RAM moyenne est la même dans les 4 gammes de prix.
# H1 : au moins une gamme a une RAM moyenne différente.

# df[df["price_range"] == k] : garde seulement les lignes de la classe k (filtre booléen)
# ["ram"] : puis on prend leur RAM ; range(4) : k = 0,1,2,3
groups = [df[df["price_range"] == k]["ram"] for k in range(4)]
# stats.f_oneway(*groups) : l'ANOVA sur les 4 groupes (* = "déballe" la liste en arguments)
# renvoie la statistique F et la p-value
f_stat, p_value = stats.f_oneway(*groups)

# df.groupby("price_range")["ram"].mean() : RAM moyenne PAR gamme de prix
print("Mean RAM per price range:")
print(df.groupby("price_range")["ram"].mean().round(1))
print("\nANOVA  F = %.2f   p = %.3e" % (f_stat, p_value))

# Si p-value < 0.05 -> différence significative (on rejette H0)
if p_value < 0.05:
    print("=> Reject H0: RAM differs significantly across price ranges.")
else:
    print("=> Fail to reject H0.")

In [ ]:
# TEST D'HYPOTHÈSE 2 - test t (compare les moyennes de DEUX groupes)
# On compare la RAM des téléphones les moins chers (0) et les plus chers (3).

# Filtre booléen : RAM de la classe 0, puis RAM de la classe 3
ram_low  = df[df["price_range"] == 0]["ram"]
ram_high = df[df["price_range"] == 3]["ram"]
# stats.ttest_ind : test t à 2 échantillons indépendants -> statistique t + p-value
t_stat, p_value = stats.ttest_ind(ram_low, ram_high)

# .mean() : moyenne de chaque groupe ; round(.., 1) : arrondi à 1 décimale
print("Mean RAM  low =", round(ram_low.mean(), 1), " | very high =", round(ram_high.mean(), 1))
print("t = %.2f   p = %.3e" % (t_stat, p_value))

if p_value < 0.05:
    print("=> Significant difference (reject H0).")
else:
    print("=> No significant difference.")

In [ ]:
# df.select_dtypes(include=[np.number]) : ne garde que les colonnes numériques
numeric_df = df.select_dtypes(include=[np.number])
# .corr() : matrice de corrélation entre toutes les colonnes
# ["price_range"] : on prend la colonne des corrélations avec la cible
# .drop("price_range") : on enlève la corrélation de la cible avec elle-même (= 1)
corr_with_target = numeric_df.corr()["price_range"].drop("price_range")
# .sort_values(ascending=False) : tri du plus corrélé au moins corrélé
print(corr_with_target.sort_values(ascending=False).round(3))

# stats.pearsonr : renvoie le coefficient de corrélation ET sa p-value
r, p_value = stats.pearsonr(df["ram"], df["price_range"])
print("\nPearson r (ram vs price_range) = %.3f   p = %.3e" % (r, p_value))

## 4. Visualisation des données avec Matplotlib

In [ ]:
# features : les variables dont on veut voir la distribution
features = ["ram", "battery_power", "px_width", "int_memory"]
# plt.figure(figsize=(12,8)) : crée une grande figure (12 de large, 8 de haut)
plt.figure(figsize=(12, 8))
# enumerate(features, 1) : parcourt la liste en numérotant à partir de 1
for i, col in enumerate(features, 1):
    # plt.subplot(2,2,i) : place le i-ème graphique dans une grille 2x2
    plt.subplot(2, 2, i)
    # plt.hist : histogramme ; bins=30 : 30 barres ; edgecolor : contour noir
    plt.hist(df[col], bins=30, edgecolor="black")
    plt.title(col)          # titre = nom de la variable
    plt.xlabel(col)
    plt.ylabel("Count")
# plt.tight_layout() : ajuste les espacements pour ne rien chevaucher
plt.tight_layout()
plt.show()                  # affiche la figure

In [ ]:
# data_by_class : la RAM de chaque classe de prix (liste de 4 séries)
data_by_class = [df[df["price_range"] == k]["ram"] for k in range(4)]
plt.figure(figsize=(10, 6))
# plt.boxplot : boîte à moustaches (médiane, quartiles, extrêmes) par groupe
plt.boxplot(data_by_class)                    # positions par défaut = 1..4
# plt.xticks : remplace les positions 1..4 par des étiquettes lisibles
plt.xticks([1, 2, 3, 4], ["Low", "Medium", "High", "Very High"])
plt.title("RAM distribution by price range")
plt.xlabel("Price range")
plt.ylabel("RAM (MB)")
plt.grid(True, axis="y")   # grille horizontale seulement
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# plt.scatter : nuage de points (x = batterie, y = RAM)
# c=df["price_range"] : la couleur dépend de la gamme de prix
# cmap="viridis" : palette de couleurs ; alpha=0.6 : points semi-transparents
sc = plt.scatter(df["battery_power"], df["ram"], c=df["price_range"],
                 cmap="viridis", alpha=0.6)
# plt.colorbar : ajoute l'échelle de couleurs à droite
plt.colorbar(sc, label="Price range")
plt.title("Battery power vs RAM (colour = price range)")
plt.xlabel("Battery power (mAh)")
plt.ylabel("RAM (MB)")
plt.tight_layout()
plt.show()

In [ ]:
# numeric_df : uniquement les colonnes numériques ; corr : matrice de corrélation
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

# plt.subplots : crée une figure (fig) et une zone de dessin (ax)
fig, ax = plt.subplots(figsize=(12, 10))
# ax.imshow : affiche la matrice comme une image colorée (la "carte thermique")
# cmap="coolwarm" : bleu = négatif, rouge = positif ; vmin/vmax : bornes -1 à 1
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
# set_xticks / set_xticklabels : place et nomme les graduations (les variables)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)   # rotation=90 : noms à la verticale
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
# fig.colorbar : la légende de couleurs (de -1 à 1)
fig.colorbar(im, label="Correlation")
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# On recalcule corr ici pour que la cellule soit autonome (indépendante des autres)
corr = df.select_dtypes(include=[np.number]).corr()
# corr["price_range"] : corrélations avec la cible ; .drop : on retire la cible elle-même
# .sort_values() : tri croissant (pour un classement horizontal lisible)
corr_sorted = corr["price_range"].drop("price_range").sort_values()
plt.figure(figsize=(10, 7))
# plt.barh : diagramme en barres HORIZONTALES (h = horizontal)
plt.barh(corr_sorted.index, corr_sorted.values, color="teal")
plt.title("Correlation of each feature with price_range")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## 5. Synthèse et conclusion

**Qualité du jeu de données**
- 2 000 téléphones, 20 caractéristiques, 1 cible. Aucune valeur manquante, aucun doublon,
  et les quatre classes de prix sont parfaitement équilibrées (500 chacune).

**Le driver dominant : la RAM**
- La RAM est corrélée à **0,92** avec la gamme de prix - de loin la relation la plus forte.
- La RAM moyenne augmente régulièrement avec le prix : **785 -> 1 680 -> 2 583 -> 3 449 MB**.
- L'ANOVA (F = 3520, p ~ 0) et le test t entre les téléphones les moins chers et les plus
  chers (t = -111, p ~ 0) confirment tous deux une différence hautement significative.

**Drivers secondaires**
- `battery_power` (0,20), `px_width` (0,17) et `px_height` (0,15) montrent des corrélations
  positives faibles mais réelles avec le prix.
- Toutes les autres variables (cœurs, caméras, indicateurs de connectivité, poids,
  épaisseur...) sont quasiment non corrélées au prix (|r| < 0,05).

**Formes des distributions**
- La plupart des variables sont quasi uniformes (skew ~ 0, kurtosis ~ -1,2), typique de ce
  jeu de données synthétique. `fc`, `px_height` et `sc_w` sont asymétriques à droite.

**Découverte inattendue**
- Des specs intuitivement "haut de gamme" comme le nombre de cœurs, la fréquence du
  processeur et les mégapixels des caméras n'influencent presque pas la classe de prix -
  **la taille de la mémoire domine tout**.

**Conclusion**
- Pour classer la gamme de prix d'un téléphone, **la RAM est de loin la caractéristique la
  plus importante**, avec la batterie et la résolution d'écran comme contributeurs mineurs.
  Un modèle basé sur la seule RAM séparerait déjà très bien les classes.